In [ ]:
import pandas as pd

# Загрузка и предобработка BoM
def load_bom(file_path):
    df = pd.read_excel(file_path)
    
    # Приведение колонок к lowercase
    df.columns = df.columns.str.lower()
    
    # Переименование колонок
    df.rename(columns={
        'plant_id': 'plant',
        'produced_material': 'prod_material_id',
        'produced_material_release_type': 'prod_material_release_type',
        'produced_material_production_type': 'prod_material_production_type',
        'produced_material_quantity': 'prod_material_quantity',
        'component_material': 'component_id',
        'component_material_release_type': 'component_material_release_type',
        'component_material_production_type': 'component_material_production_type',
        'component_material_quantity': 'component_consumption_quantity',
        'year': 'year',
        'month': 'month'
    }, inplace=True)
    
    return df

# Рекурсивный BoM explosion
def explode_bom(fin_id, df, plant, year):

    # Рекурсивная цепочка BoM для одного FIN материала
    result_rows = []
    df_filtered = df[(df['plant'] == plant) & (df['year'] == year)]
    
    if df_filtered[df_filtered['prod_material_id'] == fin_id].empty:
        return pd.DataFrame()
    
    fin_row = df_filtered[df_filtered['prod_material_id'] == fin_id].iloc[0]
    
    def recurse(material_id):
        rows = df_filtered[df_filtered['prod_material_id'] == material_id]
        if rows.empty:
            return
        
        for _, row in rows.iterrows():
            result_rows.append({
                'plant': plant,
                'fin_material_id': fin_id,
                'fin_material_release_type': fin_row['prod_material_release_type'],
                'fin_material_production_type': fin_row['prod_material_production_type'],
                'fin_production_quantity': fin_row['prod_material_quantity'],
                'prod_material_id': row['prod_material_id'],
                'prod_material_release_type': row['prod_material_release_type'],
                'prod_material_production_type': row['prod_material_production_type'],
                'prod_material_production_quantity': row['prod_material_quantity'],
                'component_id': row['component_id'],
                'component_material_release_type': row['component_material_release_type'],
                'component_material_production_type': row['component_material_production_type'],
                'component_consumption_quantity': row['component_consumption_quantity'],
                'year': year
            })
            
            # Рекурсия для FIN/PROD компонентов
            if row['component_material_release_type'] in ['FIN','PROD'] and row['component_id'] in df_filtered['prod_material_id'].values:
                recurse(row['component_id'])
    
    recurse(fin_id)
    return pd.DataFrame(result_rows)

# Логика по заводу и году
def bom_explosion(df):
    all_results = []
    
    for plant in df['plant'].unique():
        years_for_plant = df[df['plant'] == plant]['year'].unique()
        for year in years_for_plant:
            # FIN материалы для данного завода и года
            fin_materials = df[
                (df['plant'] == plant) & 
                (df['year'] == year) & 
                (df['prod_material_release_type'] == 'FIN')
            ]['prod_material_id'].unique()
            
            for fin_id in fin_materials:
                exploded = explode_bom(fin_id, df, plant, year)
                if not exploded.empty:
                    all_results.append(exploded)
    
    if all_results:
        exploded_df = pd.concat(all_results, ignore_index=True)
        # Сортировка
        exploded_df = exploded_df.sort_values(by=['plant', 'year', 'fin_material_id', 'prod_material_id', 'component_id'])
        return exploded_df
    else:
        return pd.DataFrame()


# Использование 
file_path = 'task_2_data_ex.xlsx'
df_bom = load_bom(file_path)

exploded_df = bom_explosion(df_bom)

# Сохранение результата
exploded_df.to_csv('task_2.csv', index=False)
print("BoM explosion готов, сохранен в 'task_2.csv'")